
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Creating and Curating A Knowledge Assistant
## Overview
In this lab, you'll create your own Knowledge Assistant, attach two shared AI Search indexes as knowledge sources (one programmatically via `IndexSpec` and one through the UI), and test the assistant both in the Playground and from a notebook. You'll then improve response quality by refining the assistant's instructions and set up an LLM judge to evaluate completeness using Databricks built-in scorers.

## Learning Objectives
By the end of this lab, you will be able to:
1. Create a Knowledge Assistant programmatically using the Databricks SDK
2. Attach a AI Search index as a knowledge source using `IndexSpec`
3. Attach an additional knowledge source via the UI
4. Query a Knowledge Assistant programmatically using the OpenAI client
5. Improve response quality by refining assistant instructions
6. Create an LLM judge to evaluate trace quality

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Beta Features</strong>
<p style="margin: 8px 0 0 0; color: #333;">This notebook uses Databricks Beta Features. While this notebook has been thoroughly tested, it's worth noting that full functionality is not guaranteed.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
After running the classroom setup, you will need to define a few variables that you will use to create your [Knowledge Assistant using the Databricks SDK](https://docs.databricks.com/api/workspace/knowledgeassistants).

In this lab, you will use two **shared AI Search indexes** that were created during the workspace setup:

- **`dbacademy.ka_all.house_rules_index`**: Built by parsing `airbnb_house_rules.pdf` with `ai_parse_document`, chunking semantically with `ai_prep_search`, and indexing the chunks with managed embeddings (`databricks-gte-large-en`).
- **`dbacademy.ka_all.airbnb_listings_index`**: Built by parsing `airbnb_listings.pdf` with `ai_parse_document`, chunking by property blocks, and indexing with the same embedding model.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Shared Indexes</strong>
<p>Both indexes live in the shared <code>dbacademy.ka_all</code> catalog/schema on the <code>vs_all_demo</code> endpoint. You do not need to create them. They are ready to attach as knowledge sources to your Knowledge Assistant.</p>
</div>

In [0]:
%run ./Includes/Classroom-Setup-08

In [0]:
# Shared VS indexes (created during workspace setup)
HOUSE_RULES_VS_INDEX = "dbacademy.ka_all.house_rules_index"
LISTINGS_VS_INDEX = "dbacademy.ka_all.airbnb_listings_index"

print(f"House Rules VS Index: {HOUSE_RULES_VS_INDEX}")
print(f"Listings VS Index:    {LISTINGS_VS_INDEX}")

## B. Verify the Shared AI Search Indexes

Before creating the Knowledge Assistant, let's confirm the shared AI Search indexes are ready.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()
VS_ENDPOINT = "vs_all_demo"

for idx_name in [HOUSE_RULES_VS_INDEX, LISTINGS_VS_INDEX]:
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=idx_name)
    status = index.describe().get("status", {}).get("detailed_state")
    print(f"{idx_name}: {status}")

## C. Create a Knowledge Assistant

Create a new Knowledge Assistant with a descriptive name and instructions. This will be **your** assistant, separate from the workspace-level one. This is enforced by using your catalog (your username) in the name of the KA as shown in the next cell. 


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Note</strong>
<p>If you rerun this cell, you will see an <code>AlreadyExists</code> runtime error. In this case, you may move onto the next cell since your KA has already been created.</p>
</div>

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.knowledgeassistants import KnowledgeAssistant

w = WorkspaceClient()

# TODO: Create your Knowledge Assistant
ka = KnowledgeAssistant(
    display_name=f"{catalog_name}-ka",   # e.g., "labuser1234_567-ka" 
    description="<FILL_IN>",   # e.g., "Answers questions about Airbnb listings and house rules"
    instructions="<FILL_IN>",  # e.g., "You are a helpful assistant that answers questions about Airbnb listings..."
)

created_ka = w.knowledge_assistants.create_knowledge_assistant(knowledge_assistant=ka)
print(f"Knowledge Assistant created!")
print(f"  Name: {created_ka.display_name}")
print(f"  ID:   {created_ka.name}")


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C: Create a Knowledge Assistant
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.knowledgeassistants import KnowledgeAssistant

w = WorkspaceClient()

ka = KnowledgeAssistant(
    display_name=f"{catalog_name}-ka",
    description="Answers questions about Airbnb listings and house rules in San Francisco",
    instructions="You are a helpful assistant that answers questions about San Francisco Airbnb listings, house rules, and property details. Always cite the source documents when providing information. If the answer is not in the provided data, say so.",
)

created_ka = w.knowledge_assistants.create_knowledge_assistant(knowledge_assistant=ka)
print(f"Knowledge Assistant created!")
print(f"  Name: {created_ka.display_name}")
print(f"  ID:   {created_ka.name}")
```

</div>
</details>

## D. Attach House Rules Index via Code (IndexSpec)

Attach the shared house rules AI Search index as a knowledge source using `IndexSpec`. This gives the assistant access to the house rules data.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Reference</strong>
<p>See the Databricks documentation on Knowledge Assistant - Manage Knowledge Sources (<a href="https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant?language=Vector%C2%A0Search%C2%A0Index#manage-knowledge-sources-1" style="color: #1976d2;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-bricks/knowledge-assistant?language=Vector%C2%A0Search%C2%A0Index#manage-knowledge-sources-1" style="color: #1976d2;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/agent-bricks/knowledge-assistant?language=Vector%C2%A0Search%C2%A0Index#manage-knowledge-sources-1" style="color: #1976d2;">GCP</a>) for details on attaching AI Search indexes programmatically.</p>
</div>

In [0]:
from databricks.sdk.service.knowledgeassistants import (
    KnowledgeSource,
    IndexSpec,
)

# TODO: Attach the house rules VS index as a knowledge source
house_rules_source = KnowledgeSource(
    display_name="House Rules Index",
    description="AI search index over Airbnb house rules and host information",
    source_type=<FILL_IN>,        # Set to a AI search index
    index=IndexSpec(
        index_name=<FILL_IN>,     # Your VS index name
        text_col=<FILL_IN>,       # The column containing the chunk text
        doc_uri_col=<FILL_IN>,    # The column containing the source document path
    ),
)

created_source_1 = w.knowledge_assistants.create_knowledge_source(
    parent=created_ka.name,
    knowledge_source=house_rules_source,
)
print(f"Knowledge Source added: {created_source_1.display_name}")


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D: Attach House Rules Index via Code
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from databricks.sdk.service.knowledgeassistants import (
    KnowledgeSource,
    IndexSpec,
)

house_rules_source = KnowledgeSource(
    display_name="House Rules Index",
    description="AI search index over Airbnb house rules and host information",
    source_type="index",
    index=IndexSpec(
        index_name=HOUSE_RULES_VS_INDEX,
        text_col="chunk_text",
        doc_uri_col="source_path",
    ),
)

created_source_1 = w.knowledge_assistants.create_knowledge_source(
    parent=created_ka.name,
    knowledge_source=house_rules_source,
)
print(f"Knowledge Source added: {created_source_1.display_name}")
```

</div>
</details>

## E. Attach Listings Index via the UI

Now let's add the workspace-level Airbnb listings index via the Databricks UI. This gives the assistant a second knowledge source with property descriptions.

Follow these steps:

1. In the left sidebar, navigate to **Agents**
2. Find your Knowledge Assistant (the one you just created) and click on it
3. Click on **+ Add** under the already existing Index
4. In **Type** select **AI Search Index** and search `dbacademy.ka_all.airbnb_listings_index`
5. The name will automatically be filled out as `airbnb_listings_index`
6. For the **Doc URI Column**, select **source_path**
7. For **Text Column**, select **chunk_text**
8. For **Describe the content**, copy and paste the following: 
<div class="code-block-light" data-language="text">
AI search index over parsed Airbnb listing descriptions
</div>

<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>


9. Select **Save and Update**

Once added, your Knowledge Assistant will have **two knowledge sources**: house rules (via code) and listings (via UI).

Run the next cell to verify the knowledge sources are attached.

In [0]:
# Verify both knowledge sources are attached
sources = w.knowledge_assistants.list_knowledge_sources(parent=created_ka.name)
print(f"Knowledge Sources for '{created_ka.display_name}':")
for source in sources:
    print(f"  - {source.display_name} ({source.source_type})")

## F. Test in the Playground

Now let's test your Knowledge Assistant. You can do this in two ways:

**Option A: Playground UI**
1. Go to **Playground** in the left sidebar
2. Select your Knowledge Assistant from the model dropdown
3. Try questions that span both knowledge sources:
<div class="code-block-light" data-language="text">
What are the house rules for properties near a park?
</div>
<div class="code-block-light" data-language="text">
Tell me about pet-friendly listings and their rules
</div>
<div class="code-block-light" data-language="text">
Which properties have strict check-in policies?
</div>

<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

### F1. Notebook Query

Fill out the `<FILL_IN>` for each section as usual, but to fill in the value for `ka_endpoint` you can find this with two methods:
#### Option 1
1. Navigate to **Serving** on the left
2. Toggle **Owned by me** in the filter menu
3. Click on the KA name (e.g. **ka-1234ab5c-endpoint**)
4. Copy the serving endpoint name and paste it down below for `ka_endpoint` (e.g. `ka_endpoint = ka-1234ab5c-endpoint`)
#### Option 2
1. Run the next cell and copy the output. The next cell uses `WorkspaceClient()` to programmatically retrieve the value for `ka_endpoint`.

In [0]:
assistants = w.knowledge_assistants.list_knowledge_assistants()
for assistant in assistants:
    if assistant.display_name.startswith(f"{catalog_name}-"):
        ka_endpoint = f"ka-{assistant.name.split("/", 1)[1].split("-")[0]}-endpoint"
        print(ka_endpoint)

In [0]:
# TODO: Query your Knowledge Assistant programmatically
from openai import OpenAI
import mlflow

mlflow.openai.autolog()

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

host = w.config.host

client = OpenAI(
    api_key=<FILL_IN>,
    base_url=<FILL_IN>,
)

ka_endpoint = <FILL_IN>
print(f"Using endpoint: {ka_endpoint}")

response = client.responses.create(
    model=ka_endpoint,
    input = [
        {
            "role": "user",
            "content": "What are the house rules for listings that mention being family-friendly?"
        }
    ]
)
print(" ".join(getattr(content, "text", "") for output in response.output for content in getattr(output, "content", [])))


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task F: Query the Knowledge Assistant
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from openai import OpenAI
import mlflow

mlflow.openai.autolog()

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = w.config.host

client = OpenAI(
    api_key=token,
    base_url=f"{host}/serving-endpoints",
)
assistants = w.knowledge_assistants.list_knowledge_assistants()
for assistant in assistants:
    if assistant.display_name.startswith(f"{catalog_name}-"):
        ka_endpoint = f"ka-{assistant.name.split("/", 1)[1].split("-")[0]}-endpoint"
        print(ka_endpoint)

print(f"Using endpoint: {ka_endpoint}")

response = client.responses.create(
    model=ka_endpoint,
    input = [
        {
            "role": "user",
            "content": "What are the house rules for listings that mention being family-friendly?"
        }
    ]
)

print(" ".join(getattr(content, "text", "") for output in response.output for content in getattr(output, "content", [])))
```

</div>
</details>

## G. Improve Quality

KAs can be improved through their instructions. General guidance on this topic falls outside the scope of the course. However, you can try the following:

1. Go to your Knowledge Assistant in the UI
2. Edit the **Instructions** to be more specific by navigating to the Agent and clicking on **Settings** at the top right. For example:
   - Add grounding rules: *"Only answer based on the provided knowledge sources. If the answer is not in the data, say so."*
   - Add formatting rules: *"Always cite the source document when providing information."*
   - Add persona details: *"You are a San Francisco vacation rental expert."*
3. Test again in the Playground and observe how the responses change

## H. LLMs as Judges
Not only can you improve the quality through general instructions like mentioned in the previous section, but we can also build LLM judges. This course does focus on [Agent Evaluation](https://www.databricks.com/training/catalog?search=evaluation), but luckily using an LLM as a judge is easy.
1. Navigate into **Experiments** on the left
2. Locate your KA's experiment (e.g. it will look something like **ka-1ab234c5-dev-experiment**) and click on it
    - Click "Owned by me" to filter to your experiment
3. By completing the test in the playground (see instructions above), you have already populated your agent with some traces. Click on **Judges** on the left under **Evaluation**
4. Click on **+ New LLM judge**
5. Under **General** select **Traces** and give it the name **completeness_judge**
6. Under **Evaluation criteria**, select the dropdown menu for **LLM judge** and select **Completeness**. Notice that **Instructions** will automatically be filled out. 
7. In the pannel to the right, click on **Select traces**
8. Select the first trace in the next window and confirm the selection by clicking **Select** at the bottom right
9. Back in the **Create LLM judges** overview page you will now have the option to **Run judge**. Select it and view the output. 
10. Click **Create judge**. This will create a Lakeflow job. 
11. Navigate to **Jobs and Pipeliness** on the left and find the job with **Trace Metrics Computation Job** in the title.
    - This is a scheduled job that runs on the full trace. You should see it running currently (depending on how quickly you moved onto this step). 
    - Open the job and click **delete** under **Schedules & Triggers** on the right side menu. 
    - If you click on the run you will see the Python wheel output, which is completing the monitoring task. 
12. Navigate back to the MLflow experiment and see that there is now an assessment column in the **Traces** menu called **completeness_judge**.
13. Click on the trace and under **Feedback** you will see the single judge has ran. Click on the dropdown under **completeness_judge** and view the LLM's feedback.

Built-in LLM judges are predefined [scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/scorers) that use Databricks-hosted LLMs to evaluate common quality dimensions of your GenAI application such as relevance, safety, groundedness, and correctness. Use them when you want to start evaluating quality quickly. For situations where you want more control over your judges, use [custom LLM judges](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/) or Python ([code-based scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers)).

## I. Clean Up Your Resources
1. Once you are finished with your knowledge assistant, navigate to **Agents** and delete _your_ agent (e.g. **labuserXXX_XXX-ka**). **Do not delete any other knowledge assistants**. Deleting the agent automatically deletes the associated KA assets (MLflow experiment, etc.), but not the job.
2. Navigate to **Jobs & Pipelines** and delete the created job.

## J. Conclusion
In this lab, you:

1. **Created** a Knowledge Assistant programmatically using the Databricks SDK
2. **Attached** the shared house rules AI Search index as a knowledge source via `IndexSpec` (code)
3. **Attached** the shared Airbnb listings index as a second knowledge source via the UI
4. **Tested** the assistant in the Playground and programmatically from a notebook
5. **Improved** quality by refining the assistant's instructions
6. **Evaluated** trace quality by creating an LLM judge using built-in scorers

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
